# Query Masking Ablations

This notebook recreates the query maskings for different random seeds and computes the overlap between them.

In [6]:
import numpy as np
from pathlib import Path

# Set parameters
frac = 0.05  # Fraction of maskable tokens to replace (adjust as needed)
seeds = [0, 1]  # Hardcoded list of seeds
fasta_file = Path("../data/ZnT8/ZnT8.fasta")  # Path to the unmasked sequence

In [7]:
# Read the original unmasked sequence
lines = fasta_file.read_text().splitlines()
unmasked_seq = ""
for i in range(0, len(lines), 2):
    if lines[i].startswith(">"):
        unmasked_seq = lines[i + 1].strip()
        break

print(f"Unmasked sequence length: {len(unmasked_seq)}")

Unmasked sequence length: 369


In [8]:
# Re-create maskings
ACID_TOKENS = set("ABCDEFGHIJKLMNOPQRSTUVWXYZ-")

def get_maskable_indices(query_seq):
    return [
        k for k, char in enumerate(query_seq)
        if char in ACID_TOKENS and char not in ("-", "X")
    ]

maskable_indices = get_maskable_indices(unmasked_seq)
n_mask = int(len(maskable_indices) * frac)

masked_sequences = {}        # Maps seed -> masked sequence (string)
masked_positions = {}        # Maps seed -> set of masked indices
visualization_sequences = {} # Maps seed -> sequence with '#' for masked positions

for seed in seeds:
    rng = np.random.default_rng(seed)
    
    chosen = []
    if n_mask > 0:
        chosen = rng.choice(maskable_indices, n_mask, replace=False)
        
    # Create actual masked sequence (Alanine substitution)
    seq_list = list(unmasked_seq)
    for k in chosen:
        seq_list[k] = "A"
    masked_sequences[seed] = "".join(seq_list)
    
    # Create visualization sequence (with '#')
    vis_list = list(unmasked_seq)
    for k in chosen:
        vis_list[k] = "#"
    visualization_sequences[seed] = "".join(vis_list)
    
    masked_positions[seed] = set(chosen)

In [9]:
# Calculate overlaps
if not seeds:
    overlap_positions = set()
else:
    overlap_positions = set(masked_positions[seeds[0]])
    for s in seeds[1:]:
        overlap_positions.intersection_update(masked_positions[s])

print(f"Number of overlaps for seeds {seeds}: {len(overlap_positions)}")

Number of overlaps for seeds [0, 1]: 2


In [10]:
# Display alignment
print("1) Original query:")
print(unmasked_seq)
print("\n" + "="*80 + "\n")

print("2) Masked sequences:")
for seed in seeds:
    print(f"Seed {seed}:")
    print(visualization_sequences[seed])
    print()
print("="*80 + "\n")

print("3) Overlap:")
overlap_str = ["-"] * len(unmasked_seq)
for pos in overlap_positions:
    overlap_str[pos] = "#"
print("".join(overlap_str))

1) Original query:
MEFLERTYLVNDKAAKMYAFTLESVELQQKPVNKDQCPRERPEELESGGMYHCHSGSKPTEKGANEYAYAKWKLCSASAICFIFMIAEVVGGHIAGSLAVVTDAAHLLIDLTSFLLSLFSLWLSSKPPSKRLTFGWHRAEILGALLSILCIWVVTGVLVYLACERLLYPDYQIQATVMIIVSSCAVAANIVLTVVLHQRCLGHNHKEVQANASVRAAFVHALGDLFQSISVLISALIIYFKPEYKIADPICTFIFSILVLASTITILKDFSILLMEGVPKSLNYSGVKELILAVDGVLSVHSLHIWSLTMNQVILSAHVATAASRDSQVVRREIAKALSKSFTMHSLTIQMESPVDQDPDCLFCEDPCD


2) Masked sequences:
Seed 0:
MEFLE#TYLVNDKA#KMYAFTLESVE#QQKPVNKDQCPRERPEELESGGMYHCHSGSKPTEKG#NEYAYAKWKLCSASAICFIFMIAEVVGGHIA#SLAVVTDAAHLLI#LTSFLLSLFSLWLSSKPPSKRLTFGWHRAEILGALLSILCIWVVTGVLVYLACERLLYPDYQIQATVMII#SS#AVAANIVLTVVLHQRC#GHNHKEVQANASVRAAFVHA#GD#FQSISVL#SA#IIYFKPEYKIADPICTFIFSILVLASTITIL#DFSILLMEGVPKSLNYSGVKELILA#DGVLS#HSLHIWSLTMNQVILSAHVATAASRDSQVVR#EIAKALSKSFTMHSLTIQMESPV#QDPDCLFCEDPCD

Seed 1:
MEFLERTYLVND#AAKMYAFTLESVELQQKPVNKDQCPRERPEELESGGMY#CHSGSKPTEKGANEYAYAKWKLCSASAICFIFMIAEV#GGHI#GSLA#VTDAAHLLIDLT#FLLSLFSLWLSSKPPSKRLTFGWHRAEILGALLSILC#WV#TGVLVYLACERL#YPDYQIQATVMII#SSCAVAANIVLTVVLHQRCL